# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook guides you in loading, exploring, and processing the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library. All dataset elements are referenced by their `@id` for reproducibility.

### Dataset Source
Croissant Schema URL: [https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Extract metadata as an object
metadata = dataset.metadata

# Display dataset name and description
print(f"{metadata.name}: {metadata.description}")

# Show publication date and source identifier
print(f"Published: {metadata.datePublished}")
print(f"Identifier: {metadata.identifier}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Enumerate all available record sets by their @id
record_sets = dataset.record_sets

print("Record sets in the dataset:")
for rs in record_sets:
    print(f"RecordSet @id: {rs['@id']}, name: {rs.get('name', 'N/A')}")

# Display fields for each record set
for rs in record_sets:
    print(f"\nFields for RecordSet @id: {rs['@id']}")
    if 'field' in rs:
        for f in rs['field']:
            print(f"  Field @id: {f['@id']}, name: {f.get('name', 'N/A')}, type: {f.get('dataType', 'N/A')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All access is via `@id` references.

In [ ]:
# List the @id of the main record set (typically survey or main data)
# Let's assume the main record set's @id is 'https://api.app.sen.science/frontiers/7160411/mental_health_survey_data'
# If unclear, use the first record set for demonstration.

record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    # Load records from each record set
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"RecordSet @id: {record_set_id}, Columns: {df.columns.tolist()}")

# Pick the first loaded record set for further analysis
main_record_set_id = record_set_ids[0]
print(f"\nPreviewing first 5 records from RecordSet @id: {main_record_set_id}")
if main_record_set_id in dataframes:
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering records, normalizing numeric fields, categorizing data, removing outliers, and grouping for insights.

> All columns/fields are referenced by their `@id`. If multiple numeric fields are present, select one for demonstration (e.g., GAD-7 score, PHQ-9 score).

In [ ]:
# Identify a numeric field by @id for EDA
# Assume GAD-7 score column @id (replace with correct one from overview)
# For the demo, use column names from DataFrame
numeric_field = None
if main_record_set_id in dataframes:
    df = dataframes[main_record_set_id]
    # Try to auto-select a GAD-7/PHQ-9 column (fallback to any numeric field)
    for col in df.columns:
        if "GAD" in col or "PHQ" in col or "score" in col:
            numeric_field = col
            break
    if numeric_field is None:
        numeric_field = df.select_dtypes(include='number').columns[0] if not df.select_dtypes(include='number').empty else df.columns[0]

    print(f"Using numeric field for EDA: {numeric_field}")

    # Filter records above a threshold
    threshold = 10
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"].head()])

    # Group by another field (e.g., 'gender' or 'village'), using @id or column name
    group_field = None
    for col in df.columns:
        if "gender" in col.lower() or "village" in col.lower():
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by {group_field}:")
        display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Here, we plot the distribution of the selected numeric field and its relationship with the grouping attribute, if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id in dataframes and numeric_field:
    df = dataframes[main_record_set_id]
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    if group_field:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()

## 6. Conclusion
This notebook successfully loaded and explored the FAIR² Mental Health Survey Dataset using the `mlcroissant` library. By referencing all entities via their `@id`, we ensured reproducibility and traceability.

- Dataset contains survey data with rich demographic and mental health indicators.
- We demonstrated data loading, overview, EDA (filtering, normalization, grouping), and basic visualizations.
- The structure defined by Croissant provides explicit data lineage for every field and record set.

> For further analysis, consider exploring relationships between assessment scores and demographic variables, or benchmarking against population-level mental health statistics.